# PoliMillionaire Game Tests

Run selected local models one after another.

## Setup

In [1]:
import gc
import importlib.util
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))
print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])

Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Set `run=True` for the models you want to test.

In [2]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_IDS = [0, 1, 2, 3, 4, 5]
SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "kind": "gemma", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "kind": "gemma", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen3.5 2B Thinking", "kind": "qwen_thinking", "model_id": "Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B Council", "kind": "gemma_council", "model_id": "google/gemma-4-E2B-it", "votes": 3, "run": False},
    {"label": "Gemma + Qwen Mixed Council", "kind": "mixed_council", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B (4-bit)", "kind": "gemma_quantized", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma + Qwen Mixed Council (4-bit)", "kind": "mixed_quantized", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B LangChain Agent (4-bit)", "kind": "langchain_agentic", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E2B + Routed RAG", "kind": "rag", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma + Qwen Mixed Council + Routed RAG (4-bit)", "kind": "mixed_quantized_rag", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Mistral 7B (4-bit)", "kind": "llama_cpp_mistral_quantized", "model_id": "Mistral/Mistral-7B-Instruct-v0.1", "run": True},
]

print("API check:", RUN_API_CHECK, "Live game:", RUN_LIVE_GAME, "Benchmark:", RUN_OFFLINE_BENCHMARK)
print("Competition IDs:", COMPETITION_IDS)
for item in MODELS_TO_RUN:
    print("-", item["label"], "run=", item["run"])

API check: True Live game: True Benchmark: True
Competition IDs: [0, 1, 2, 3, 4, 5]
- Gemma 4 E2B run= False
- Gemma 4 E4B run= False
- Qwen3.5 2B Thinking run= False
- Gemma 4 E2B Council run= False
- Gemma + Qwen Mixed Council run= False
- Gemma 4 E2B (4-bit) run= False
- Gemma + Qwen Mixed Council (4-bit) run= False
- Gemma 4 E2B LangChain Agent (4-bit) run= False
- Gemma 4 E2B + Routed RAG run= False
- Gemma + Qwen Mixed Council + Routed RAG (4-bit) run= False
- Mistral 7B (4-bit) run= True


## Login

In [3]:
import getpass

from dotenv import load_dotenv
from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}"   )

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")
try:
    from google.colab import userdata
    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass
if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")
print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))

Username configured: True
Password configured: True


## Competitions

In [4]:
client = None
if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")

Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


### Use model LlamaCpp

In [5]:
from __future__ import annotations

import multiprocessing
import os
from pathlib import Path
from typing import Any

from huggingface_hub import hf_hub_download
from polimillionaire.strategies import GemmaStrategy
from polimillionaire.types import AnswerOption, Question

try:
    from llama_cpp import Llama
except Exception as exc:
    raise RuntimeError(
        "Missing llama-cpp-python. Install it first, then restart Python."
    ) from exc


DEFAULT_REPO_ID = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF"
DEFAULT_FILENAME = "mistral-7b-instruct-v0.1.Q4_K_M.gguf"

os.environ.setdefault(
    "GGUF_MODEL_PATH",
    str(REPO_ROOT / "data" / "llama-cpp" / DEFAULT_FILENAME),
)

GGUF_MODEL_PATH = Path(os.environ["GGUF_MODEL_PATH"])


def ensure_gguf_model(
    *,
    repo_id: str,
    filename: str,
    local_path: str | Path,
) -> Path:
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        print(f"Loading existing local GGUF: {local_path}")
        return local_path

    print(f"Downloading GGUF from Hugging Face: {repo_id}/{filename}")
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=str(local_path.parent),
    )

    downloaded_path = Path(downloaded_path)

    if downloaded_path != local_path and downloaded_path.exists():
        print(f"Downloaded to cache/local dir: {downloaded_path}")

    if local_path.exists():
        print(f"Using local GGUF: {local_path}")
        return local_path

    return downloaded_path


class LlamaCppLLM:
    def __init__(
        self,
        *,
        repo_id: str,
        filename: str,
        model_path: str | Path,
        max_new_tokens: int = 64,
        temperature: float = 0.0,
        n_threads: int | None = None,
        n_ctx: int = 4096,
        n_gpu_layers: int = -1,
        n_batch: int = 512,
        n_ubatch: int | None = None,
        top_p: float = 1.0,
        top_k: int = 40,
        repeat_penalty: float = 1.05,
        seed: int = 42,
        verbose: bool = False,
        model_kwargs: dict[str, Any] | None = None,
    ) -> None:
        self.repo_id = repo_id
        self.filename = filename
        self.model_path = ensure_gguf_model(
            repo_id=repo_id,
            filename=filename,
            local_path=model_path,
        )
        self.model_name = self.model_path.name

        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k
        self.repeat_penalty = repeat_penalty

        self.n_threads = n_threads or max(1, multiprocessing.cpu_count() - 1)
        self.n_ctx = n_ctx
        self.n_gpu_layers = n_gpu_layers
        self.n_batch = n_batch
        self.n_ubatch = n_ubatch if n_ubatch is not None else n_batch
        self.seed = seed
        self.verbose = verbose
        self.model_kwargs = model_kwargs or {}

        print(f"Initializing llama.cpp with model: {self.model_path}")
        self.model = Llama(
            model_path=str(self.model_path),
            n_ctx=self.n_ctx,
            n_threads=self.n_threads,
            n_gpu_layers=self.n_gpu_layers,
            n_batch=self.n_batch,
            n_ubatch=self.n_ubatch,
            seed=self.seed,
            verbose=self.verbose,
            **self.model_kwargs,
        )

    @property
    def device_summary(self) -> str:
        backend = "metal/gpu" if self.n_gpu_layers != 0 else "cpu"
        return (
            f"gguf via llama.cpp ({backend}), "
            f"threads={self.n_threads}, ctx={self.n_ctx}, batch={self.n_batch}"
        )

    def generate(self, prompt: str, **kwargs: Any) -> str:
        max_tokens = int(kwargs.get("max_new_tokens", kwargs.get("max_tokens", self.max_new_tokens)))
        temperature = float(kwargs.get("temperature", self.temperature))
        top_p = float(kwargs.get("top_p", self.top_p))
        top_k = int(kwargs.get("top_k", self.top_k))
        repeat_penalty = float(kwargs.get("repeat_penalty", self.repeat_penalty))
        stop = kwargs.get("stop")
        echo = bool(kwargs.get("echo", False))

        response = self.model.create_completion(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repeat_penalty=repeat_penalty,
            stop=stop,
            echo=echo,
        )

        try:
            return str(response["choices"][0]["text"]).strip()
        except Exception:
            return str(response).strip()

    def invoke(self, input_data: Any, config: Any = None) -> str:
        prompt = input_data if isinstance(input_data, str) else str(input_data)
        return self.generate(prompt)


llm = LlamaCppLLM(
    repo_id=DEFAULT_REPO_ID,
    filename=DEFAULT_FILENAME,
    model_path=GGUF_MODEL_PATH,
    max_new_tokens=64,
    temperature=0.0,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_batch=512,
    n_ubatch=512,
    top_p=1.0,
    top_k=40,
    repeat_penalty=1.05,
    verbose=False,
    model_kwargs={
        "use_mmap": True,
        "use_mlock": False,
    },
)

print("Model:", llm.model_name)
print("GGUF path:", llm.model_path)
print("Backend:", llm.device_summary)

sample = llm.generate(
    "Once upon a time,",
    max_tokens=64,
    temperature=0.0,
    top_p=0.9,
)
print("first warmup sample:", sample)

strategy = GemmaStrategy(llm=llm)

q = Question(
    1,
    "What is 2 + 2?",
    [
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
    ],
)

pred = strategy.answer(q)
print("Prediction:", pred.option_id, pred.answer_text, "device:", pred.metadata.get("device"))

/Users/camiloa2m/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading existing local GGUF: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/llama-cpp/mistral-7b-instruct-v0.1.Q4_K_M.gguf
Initializing llama.cpp with model: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/llama-cpp/mistral-7b-instruct-v0.1.Q4_K_M.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model: mistral-7b-instruct-v0.1.Q4_K_M.gguf
GGUF path: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/llama-cpp/mistral-7b-instruct-v0.1.Q4_K_M.gguf
Backend: gguf via llama.cpp (metal/gpu), threads=7, ctx=4096, batch=512
first warmup sample: in a land far, far away, there was a kingdom ruled by a kind and just king. The king was loved by all his subjects, and the kingdom prospered under his rule. However, one day, an evil sorcerer cast a spell on the king, causing him to fall into a deep sleep.
Prediction: 1 4 device: gguf via llama.cpp (metal/gpu), threads=7, ctx=4096, batch=512


## Run Selected Models

The 4-bit option follows the class quantization recipe and needs `bitsandbytes`.

In [6]:
# from polimillionaire.runner import GameRunner, RunLogger, load_jsonl, summarize_attempts
# from polimillionaire.strategies import CouncilStrategy, GemmaLLM, GemmaLLMConfig, GemmaStrategy, QwenLLM, QwenLLMConfig, QwenStrategy, LangChainAgenticStrategy, HeuristicStrategy, RAGStrategy, RAGConfig, RoutedStrategy
# from polimillionaire.types import AnswerOption, Question

# warmup_question = Question(1, "What is 2 + 2?", [AnswerOption(0, "3"), AnswerOption(1, "4"), AnswerOption(2, "5"), AnswerOption(3, "22")])
# rag_warmup_question = Question(4, "In which year was A Haunted House 2 released?", [AnswerOption(0, "2012"), AnswerOption(1, "2015"), AnswerOption(2, "2014"), AnswerOption(3, "2013")])
# BENCHMARK_SET = [
#     (warmup_question, 1),
#     (Question(2, "Which song was NOT written by Bob Dylan?", [AnswerOption(0, "Like a Rolling Stone"), AnswerOption(1, "Blowin' in the Wind"), AnswerOption(2, "The Times They Are A-Changin'"), AnswerOption(3, "Hound Dog")]), 3),
#     (Question(3, "What was Whitney Houston's debut album?", [AnswerOption(0, "Whitney Houston"), AnswerOption(1, "Just Whitney"), AnswerOption(2, "I'm Your Baby Tonight"), AnswerOption(3, "Whitney")]), 0),
# ]


# def clear_model_memory():
#     gc.collect()
#     try:
#         import torch
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#             torch.cuda.ipc_collect()
#     except Exception as exc:
#         print("Cleanup warning:", exc)


# def mixed_council(quantized=False):
#     if quantized and importlib.util.find_spec("bitsandbytes") is None:
#         raise RuntimeError("For the 4-bit council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
#     gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=quantized, max_new_tokens=32, do_sample=True, seed=42, generation_max_time_seconds=6.0, timeout_seconds=120.0))
#     qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=quantized, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=6.0, timeout_seconds=120.0))
#     return CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only" if quantized else "any_option", rejected_judge_fallback="primary_candidate" if quantized else "confidence_weighted", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)


# def mixed_quantized_routed_rag():
#     if importlib.util.find_spec("bitsandbytes") is None:
#         raise RuntimeError("For 4-bit mixed routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
#     gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=32, temperature=0.0, do_sample=False, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
#     qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=18.0, timeout_seconds=120.0))
#     direct = GemmaStrategy(llm=gemma)
#     council = CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only", rejected_judge_fallback="primary_candidate", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)
#     rag = RAGStrategy(llm=gemma, retrieval_config=RAGConfig(num_extra_queries=0), fallback_strategy=council)
#     return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=council, low_confidence_strategy=council, rag_min_confidence=0.65)


# def has_rag_strategy(strategy):
#     if isinstance(strategy, RAGStrategy):
#         return True
#     if isinstance(strategy, RoutedStrategy):
#         return has_rag_strategy(strategy.rag_strategy)
#     return False


# def strategy_devices(strategy):
#     devices = []

#     def collect(item):
#         if item is None:
#             return
#         if isinstance(item, CouncilStrategy):
#             for llm in item.candidate_llms + [item.judge_llm]:
#                 devices.append(getattr(llm, "device_summary", "unknown"))
#             return
#         if isinstance(item, RoutedStrategy):
#             collect(item.direct_strategy)
#             collect(item.rag_strategy)
#             collect(item.fallback_strategy)
#             return
#         if hasattr(item, "llm"):
#             devices.append(getattr(item.llm, "device_summary", "unknown"))

#     collect(strategy)
#     return list(dict.fromkeys(devices))


# def make_strategy(item):
#     if item["kind"] == "gemma_quantized":
#         if importlib.util.find_spec("bitsandbytes") is None:
#             raise RuntimeError("For 4-bit Gemma, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
#         return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], quantize_4bit=True, max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
#     if item["kind"] == "mixed_quantized":
#         return mixed_council(quantized=True)
#     if item["kind"] == "mixed_quantized_rag":
#         return mixed_quantized_routed_rag()
#     if item["kind"] == "langchain_agentic":
#         if importlib.util.find_spec("bitsandbytes") is None:
#             raise RuntimeError("For 4-bit LangChain Agent, run `%pip install -U bitsandbytes`.")
        
#         agent_config = GemmaLLMConfig(
#             model_id=item["model_id"], 
#             quantize_4bit=True, 
#             max_new_tokens=128,
#             temperature=0.0, 
#             do_sample=False, 
#             generation_max_time_seconds=30.0, 
#             timeout_seconds=120.0
#         )
#         base_llm = GemmaLLM(config=agent_config)
#         return LangChainAgenticStrategy(raw_llm=base_llm, fallback_strategy=HeuristicStrategy())
#     if item["kind"] == "mixed_council":
#         return mixed_council()
#     if item["kind"] == "gemma_council":
#         llm = GemmaLLM(GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=48, do_sample=True, seed=42, generation_max_time_seconds=4.5, timeout_seconds=120.0))
#         return CouncilStrategy(llm=llm, num_votes=item.get("votes", 3), base_seed=100, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=4.5)
#     if item["kind"] == "qwen_thinking":
#         return QwenStrategy(model_config=QwenLLMConfig(model_id=item["model_id"], max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=20, do_sample=True, seed=42, enable_thinking=True, generation_max_time_seconds=18.0, timeout_seconds=120.0))
#     if item["kind"] == "rag":
#         if importlib.util.find_spec("bitsandbytes") is None:
#             raise RuntimeError("For 4-bit RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
#         base_llm = GemmaLLM(GemmaLLMConfig(
#             model_id=item["model_id"],
#             quantize_4bit=True,
#             max_new_tokens=32,
#             temperature=0.0,
#             do_sample=False,
#             generation_max_time_seconds=18.0,
#             timeout_seconds=120.0,
#         ))
#         direct_strategy = GemmaStrategy(llm=base_llm)
#         rag_strategy = RAGStrategy(
#             llm=base_llm,
#             retrieval_config=RAGConfig(num_extra_queries=0),  # disable expansion for speed
#             fallback_strategy=HeuristicStrategy(),
#         )
#         return RoutedStrategy(direct_strategy=direct_strategy, rag_strategy=rag_strategy, fallback_strategy=direct_strategy)
#     if item["kind"] == "llama_cpp_mistral_quantized":
#         llm = LlamaCppLLM(
#             repo_id=DEFAULT_REPO_ID,
#             filename=DEFAULT_FILENAME,
#             model_path=GGUF_MODEL_PATH,
#             max_new_tokens=64,
#             temperature=0.0,
#             n_ctx=4096,
#             n_gpu_layers=-1,
#             n_batch=512,
#             n_ubatch=512,
#             top_p=1.0,
#             top_k=40,
#             repeat_penalty=1.05,
#             verbose=False,
#             model_kwargs={
#                 "use_mmap": True,
#                 "use_mlock": False,
#             },
#         )
#         return GemmaStrategy(llm=llm)
#     return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    
        


# def clean_name(label):
#     return "_".join(label.lower().replace("+", " ").replace("(", " ").replace(")", " ").split())


# benchmark_results = []


# def benchmark(strategy, label):
#     rows = []
#     for question, gold_id in BENCHMARK_SET:
#         started = time.monotonic()
#         prediction = strategy.answer(question)
#         seconds = time.monotonic() - started
#         correct = prediction.option_id == gold_id
#         rows.append((correct, seconds, bool(prediction.metadata.get("fallback"))))
#         gold = question.require_option(gold_id)
#         print("\nQ:", question.text)
#         print("predicted:", prediction.option_id, prediction.answer_text)
#         print("gold:", gold.id, gold.text, "correct:", correct, "seconds:", round(seconds, 2))
#         print("decision:", prediction.metadata.get("decision_method"), "route:", prediction.metadata.get("route"), "fallback:", prediction.metadata.get("fallback"))
#         for index, vote in enumerate(prediction.metadata.get("votes") or [], start=1):
#             print("vote", index, vote.get("model_name"), "->", vote.get("option_id"), "confidence:", vote.get("confidence"), "reason:", vote.get("reasoning"))
#         if prediction.metadata.get("judge_raw_text") is not None:
#             print("judge raw:", prediction.metadata.get("judge_raw_text"), "scope:", prediction.metadata.get("judge_scope"))
#         for source in prediction.metadata.get("evidence_sources") or []:
#             print("evidence:", source.get("title"), source.get("url"))
#     accuracy = sum(correct for correct, _, _ in rows) / len(rows)
#     max_seconds = max(seconds for _, seconds, _ in rows)
#     summary = {"model": label, "accuracy": accuracy, "avg_seconds": round(sum(seconds for _, seconds, _ in rows) / len(rows), 2), "max_seconds": round(max_seconds, 2), "fallbacks": sum(fallback for _, _, fallback in rows)}
#     benchmark_results.append(summary)
#     print("Benchmark summary:", summary)
#     return accuracy >= 0.60 and max_seconds <= 20.0 and not any(fallback for _, _, fallback in rows)


# if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
#     raise RuntimeError("Set credentials first.")
# if RUN_LIVE_GAME and client is None:
#     client = MillionaireClient(API_URL)
#     client.login(USERNAME, PASSWORD)

# run_results = []
# for item in MODELS_TO_RUN:
#     if not item.get("run", False):
#         print("Skipped", item["label"])
#         continue
#     strategy = None
#     try:
#         print("\nModel:", item["label"])
#         strategy = make_strategy(item)
#         started = time.monotonic()
#         warmup = strategy.answer(warmup_question)
#         load_and_warmup_seconds = time.monotonic() - started
#         print("warmup option:", warmup.option_id, warmup.answer_text, "fallback:", warmup.metadata.get("fallback"), "route:", warmup.metadata.get("route"), "load + answer seconds:", round(load_and_warmup_seconds, 2))
#         if warmup.metadata.get("fallback") or warmup.option_id != 1:
#             raise RuntimeError("Warm-up failed. No live game started.")
#         if item["kind"] in {"gemma_quantized", "mixed_quantized", "mixed_quantized_rag"}:
#             devices = strategy_devices(strategy)
#             print("devices:", devices)
#             if any(token in " ".join(devices).lower() for token in ("'cpu'", "'disk'", "meta")):
#                 raise RuntimeError("Quantized model was offloaded. No live game started.")
#             started = time.monotonic()
#             speed_check = strategy.answer(warmup_question)
#             answer_seconds = time.monotonic() - started
#             print("loaded-model speed check:", round(answer_seconds, 2), "seconds")
#             if speed_check.metadata.get("fallback") or speed_check.option_id != 1:
#                 raise RuntimeError("Quantized model speed check failed. No live game started.")
#             if answer_seconds > 20.0:
#                 raise RuntimeError("Quantized model loaded answer is over 20 seconds. No live game started.")
#         if has_rag_strategy(strategy):
#             started = time.monotonic()
#             rag_warmup = strategy.answer(rag_warmup_question)
#             rag_warmup_seconds = time.monotonic() - started
#             print("rag warmup option:", rag_warmup.option_id, rag_warmup.answer_text, "fallback:", rag_warmup.metadata.get("fallback"), "route:", rag_warmup.metadata.get("route"), "seconds:", round(rag_warmup_seconds, 2))
#             if rag_warmup.metadata.get("fallback") or rag_warmup.option_id != 2:
#                 raise RuntimeError("RAG warm-up failed. No live game started.")
#         benchmark_ok = benchmark(strategy, item["label"]) if RUN_OFFLINE_BENCHMARK else True
#         if item["kind"] in {"gemma_quantized", "mixed_quantized", "mixed_quantized_rag"} and RUN_LIVE_GAME and not RUN_OFFLINE_BENCHMARK:
#             raise RuntimeError("Set RUN_OFFLINE_BENCHMARK=True before a live 4-bit run.")
#         if not benchmark_ok and RUN_LIVE_GAME:
#             raise RuntimeError("Benchmark guard failed. No live game started.")
#         if not benchmark_ok:
#             print("Benchmark below live threshold; kept for offline comparison.")
#         result = {"label": item["label"], "warmup_option": warmup.option_id, "benchmark_ok": benchmark_ok}
#         if RUN_LIVE_GAME:
#             live_runs = []
#             for competition_id in COMPETITION_IDS:
#                 run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
#                 log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_competition_{competition_id}_{run_id}.jsonl"
#                 game = GameRunner(client, SAFE_DELAY_SECONDS, ANSWER_TIMEOUT_SECONDS, RunLogger(log_path)).play(competition_id, strategy)
#                 summary = summarize_attempts(load_jsonl(log_path)) if log_path.exists() else {}
#                 live_run = {"competition_id": competition_id, "earned_amount": game.earned_amount, "log_path": str(log_path), "summary": summary}
#                 live_runs.append(live_run)
#                 print("Competition:", competition_id, "Earned amount:", game.earned_amount, "Log path:", log_path)
#                 print("Summary:", summary)
#             result.update({
#                 "live_runs": live_runs,
#                 "log_paths": [run["log_path"] for run in live_runs],
#                 "earned_amount_total": sum((run.get("earned_amount") or 0) for run in live_runs),
#             })
#         run_results.append(result)
#     finally:
#         del strategy
#         clear_model_memory()
# print("\nComparison:")
# for summary in benchmark_results:
#     print(summary)
# run_results

## Speech interaction — Whisper Large V3 Turbo + LlamaCpp auto-play

In [7]:
# pip install silero-vad  # speech activity detection (no torchaudio needed at runtime; already installed)

In [8]:
import io
import re
import sys
import types
import warnings
import numpy as np
import torch
import scipy.io.wavfile as wav
import scipy.signal as signal
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from IPython.display import Audio, display

# silero-vad imports torchaudio at module level, but torchaudio is broken in this env.
# Import transformers first (it checks torchaudio availability), then stub out torchaudio
if "torchaudio" not in sys.modules:
    _mock_ta = types.ModuleType("torchaudio")
    _mock_ta.__spec__ = None
    for _sub in ("functional", "transforms"):
        _m = types.ModuleType(f"torchaudio.{_sub}")
        _m.__spec__ = None
        setattr(_mock_ta, _sub, _m)
        sys.modules[f"torchaudio.{_sub}"] = _m
    sys.modules["torchaudio"] = _mock_ta

from silero_vad import load_silero_vad, get_speech_timestamps

WHISPER_MODEL_ID = "openai/whisper-large-v3-turbo"
WHISPER_SR = 16_000
_whisper_model = None
_whisper_processor = None
_vad_model = None

# Clips shorter than this are passed directly to Whisper without VAD filtering.
# Short option clips (2-4s) have their speech spread across the whole clip;
VAD_MIN_DURATION_S = 6.0

def get_vad():
    global _vad_model
    if _vad_model is None:
        _vad_model = load_silero_vad()
    return _vad_model

def get_whisper():
    global _whisper_model, _whisper_processor
    if _whisper_model is not None:
        return _whisper_model, _whisper_processor
    print(f"Loading {WHISPER_MODEL_ID} ...")
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    _whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        WHISPER_MODEL_ID,
        dtype=dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
        cache_dir=str(REPO_ROOT / "data" / "hf_home" / "hub"),
    ).to(device)
    # suppress_tokens/begin_suppress_tokens must be None (not []) so
    # Whisper's _retrieve_logit_processors skips building those processors —
    # an empty list still triggers processor creation and causes hallucination loops.
    _whisper_model.generation_config.forced_decoder_ids = None
    _whisper_model.generation_config.suppress_tokens = None
    _whisper_model.generation_config.begin_suppress_tokens = None
    # Fully deterministic greedy decoding
    _whisper_model.generation_config.do_sample = False
    _whisper_model.generation_config.temperature = 0.0
    _whisper_model.generation_config.num_beams = 1
    _whisper_processor = AutoProcessor.from_pretrained(
        WHISPER_MODEL_ID,
        cache_dir=str(REPO_ROOT / "data" / "hf_home" / "hub"),
    )
    print(f"Whisper loaded on {device} (checkpoint max_length={_whisper_model.generation_config.max_length})")
    return _whisper_model, _whisper_processor


def _load_wav_mono_16k(audio_bytes: bytes) -> np.ndarray:
    """Read WAV bytes → float32 mono at 16 kHz."""
    src_sr, raw = wav.read(io.BytesIO(audio_bytes))
    if raw.dtype == np.int16:
        f32 = raw.astype(np.float32) / np.iinfo(np.int16).max
    elif raw.dtype == np.int32:
        f32 = raw.astype(np.float32) / np.iinfo(np.int32).max
    else:
        f32 = raw.astype(np.float32)
    if f32.ndim == 2:
        f32 = f32.mean(axis=1)
    if src_sr != WHISPER_SR:
        from scipy.signal import resample_poly
        f32 = resample_poly(f32, WHISPER_SR, src_sr)
    return f32


def _extract_speech(f32: np.ndarray, threshold: float = 0.5, pad_ms: int = 100) -> np.ndarray:
    """Use Silero VAD to keep only speech segments and discard music/silence."""
    vad = get_vad()
    pad = int(pad_ms * WHISPER_SR / 1000)
    gap = np.zeros(int(0.05 * WHISPER_SR), dtype=np.float32)
    ts = get_speech_timestamps(
        torch.from_numpy(f32), vad,
        sampling_rate=WHISPER_SR,
        threshold=threshold,
        min_speech_duration_ms=100,
        min_silence_duration_ms=200,
    )
    if not ts:
        return f32  # no speech detected, return original and let Whisper decide
    chunks = []
    for t in ts:
        s = max(0, t["start"] - pad)
        e = min(len(f32), t["end"] + pad)
        chunks.append(f32[s:e])
        chunks.append(gap)
    return np.concatenate(chunks)


def _whisper_infer(f32: np.ndarray) -> str:
    model, processor = get_whisper()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    inputs = processor(f32, sampling_rate=WHISPER_SR, return_tensors="pt")
    features = inputs["input_features"].to(device, dtype=dtype)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        ids = model.generate(
            features,
            language="english",
            task="transcribe",
            do_sample=False,
            temperature=0.0,
            num_beams=1,
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()


def _looks_like_hallucination(text: str) -> bool:
    """Detect repetition loops that indicate Whisper hallucinated."""
    if not text:
        return False
    tokens = text.replace("-", " ").split()
    if len(tokens) < 6:
        return False
    from collections import Counter
    for n in (1, 2):
        # Check for any n-gram that constitutes >35% of the text, or at least 4 occurrences.
        grams = [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        top_count = Counter(grams).most_common(1)[0][1]
        if top_count > max(4, len(grams) * 0.35):
            return True
    return False



# Punctuation/whitespace only
_PUNCT_ONLY = re.compile(r'^[\s\W]+$')

def _normalize_transcript(text: str) -> str:
    """
    Clean up Whisper output:
      - Strip surrounding quotes Whisper sometimes adds.
      - Remove punctuation-only strings (".", ",", "!").
    """
    if not text:
        return ""
    text = text.strip('"\'')
    if not text:
        return ""
    if _PUNCT_ONLY.match(text):
        return ""
    return text


def transcribe(audio_bytes: bytes) -> str:
    """
    Robust transcription for speech mixed with background music/noise:
      - Short clips (< VAD_MIN_DURATION_S): skip VAD, use raw audio directly.
        Option clips are 2-4s; VAD mangles them by splitting short speech bursts.
      - Long clips (>= VAD_MIN_DURATION_S): apply VAD to strip music-only regions,
        with fallback to raw audio if the VAD result looks like a hallucination.
      - Normalize: strip "option X" label prefix, remove punctuation-only output.
    """
    f32 = _load_wav_mono_16k(audio_bytes)
    duration_s = len(f32) / WHISPER_SR

    if duration_s < VAD_MIN_DURATION_S:
        # Short clip: VAD hurts more than it helps — go straight to Whisper
        result = _whisper_infer(f32)
        return _normalize_transcript(result)

    # Long clip: VAD-filtered path first
    speech = _extract_speech(f32)
    result = _whisper_infer(speech)

    if not _looks_like_hallucination(result):
        return _normalize_transcript(result)

    # Fallback: raw audio (VAD may have trimmed too aggressively)
    result = _whisper_infer(f32)
    if not _looks_like_hallucination(result):
        return _normalize_transcript(result)

    # Last resort: return whichever is shorter (less repetition)
    raw_result = result
    vad_result = _whisper_infer(speech)
    best = min(raw_result, vad_result, key=len) if raw_result and vad_result else (raw_result or vad_result)
    return _normalize_transcript(best)


In [9]:
from polimillionaire.types import AnswerOption, Question

def save_audio(data: bytes, filename: str):
    with open(filename, "wb") as f:
        f.write(data)
    print(f"  Saved: {filename}")

In [10]:
# Force reload with the new config (clears stale cached models)
_whisper_model = None
_whisper_processor = None
_vad_model = None
_ = get_vad()
print("Silero VAD ready.")
_ = get_whisper()
print("Whisper ready.")


Silero VAD ready.
Loading openai/whisper-large-v3-turbo ...


Loading weights: 100%|██████████| 587/587 [00:04<00:00, 128.19it/s]


Whisper loaded on mps (checkpoint max_length=448)
Whisper ready.


In [11]:
# --- ASR smoke test on saved WAVs (no server needed) ---
import glob

saved = sorted(glob.glob("level_1_*.wav") + glob.glob("notebooks/level_1_*.wav"))
if not saved:
    print("No saved level_1_*.wav files found.")
else:
    vad = get_vad()
    for path in saved:
        with open(path, "rb") as f:
            audio_bytes = f.read()

        f32 = _load_wav_mono_16k(audio_bytes)
        ts = get_speech_timestamps(
            torch.from_numpy(f32), vad,
            sampling_rate=WHISPER_SR,
            threshold=0.5,
            min_speech_duration_ms=100,
            min_silence_duration_ms=200,
        )
        total_speech_ms = sum((t["end"] - t["start"]) for t in ts) * 1000 // WHISPER_SR
        print(f"\n{path}:")
        print(f"  duration={len(f32)/WHISPER_SR:.2f}s  segments={len(ts)}  speech={total_speech_ms}ms")

        # Show raw Whisper output BEFORE normalization
        speech = _extract_speech(f32)
        raw = _whisper_infer(speech)
        print(f"  raw whisper: {raw!r}")
        print(f"  transcribe: {transcribe(audio_bytes)!r}")


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



level_1_option_A.wav:
  duration=3.68s  segments=2  speech=3066ms


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  raw whisper: 'Option A: Agurium.'
  transcribe: 'Option A: Agurium.'

level_1_option_B.wav:
  duration=2.24s  segments=1  speech=2078ms
  raw whisper: 'Option B. Atrium!'
  transcribe: 'Option B. Atrium!'

level_1_option_C.wav:
  duration=2.72s  segments=1  speech=2654ms
  raw whisper: 'Option C. Ah, ease!'
  transcribe: 'Option C. Ah, ease!'

level_1_option_D.wav:
  duration=2.32s  segments=1  speech=2254ms
  raw whisper: 'Option D: Consecratio'
  transcribe: 'Option D: Consecratio'

level_1_question.wav:
  duration=13.68s  segments=4  speech=11298ms
  raw whisper: 'That term describes the ritual act that resulted in the creation of a shrine that housed a cult image.'
  transcribe: 'That term describes the ritual act that resulted in the creation of a shrine that housed a cult image.'


In [ ]:
# def auto_play_speech(game, strategy):
#     """
#     Fully automatic speech-mode loop:
#       1. Fetch question + 4 option WAVs from the server.
#       2. Transcribe all 5 with Whisper Large V3 Turbo.
#       3. Build a Question object and ask the LLM strategy to pick the answer.
#       4. Submit before the 30-second timer expires.
#     """
#     while game.in_progress:
#         print(f"\n--- Level {game.current_level} ---")

#         # --- Fetch question audio ---
#         print("Fetching question audio...")
#         question_audio = game.fetch_audio_question()
#         save_audio(question_audio, f"level_{game.current_level}_question.wav")
#         display(Audio(question_audio))

#         # --- Fetch 4 option audios (timer starts after the last fetch) ---
#         option_audios = []
#         option_map = {}   # letter -> option_id
#         for i in range(4):
#             letter = chr(65 + i)   # A, B, C, D
#             print(f"Fetching option {letter} audio...")
#             option_audio = game.fetch_audio_option_next()
#             save_audio(option_audio, f"level_{game.current_level}_option_{letter}.wav")
#             option_audios.append(option_audio)
#             if game.current_question and i < len(game.current_question.options):
#                 option_map[letter] = game.current_question.options[i].id

#         for audio in option_audios:
#             display(Audio(audio))

#         game.refresh_state()
#         time_left = game.time_remaining
#         print(f"Time remaining after fetch: {time_left:.1f}s" if time_left else "Timer not available")

#         # --- Transcribe with Whisper ---
#         print("Transcribing question...")
#         q_text = transcribe(question_audio)
#         print(f"  Q: {q_text}")

#         option_texts = []
#         for i, audio in enumerate(option_audios):
#             letter = chr(65 + i)
#             text = transcribe(audio)
#             print(f"  {letter}: {text}")
#             option_texts.append(text)

#         # --- Build Question and ask the LLM ---
#         options = [
#             AnswerOption(id=game.current_question.options[i].id, text=option_texts[i])
#             for i in range(len(option_texts))
#         ]
#         question = Question(
#             id=game.current_question.id if game.current_question else game.current_level,
#             text=q_text,
#             options=options,
#         )

#         prediction = strategy.answer(question)
#         chosen_id = prediction.option_id
#         chosen_text = prediction.answer_text
#         print(f"LLM answer: option_id={chosen_id}  text='{chosen_text}'")

#         # --- Submit ---
#         result = game.answer(chosen_id)

#         if result.timed_out or getattr(result, "status", None) == "timeout":
#             print("TIME'S UP!")
#             print(f"Game Over. Final earnings: ${result.earned_amount:,.2f}")
#             break

#         if result.correct:
#             print("CORRECT!")
#             if result.game_over:
#                 print(f"\nCONGRATULATIONS! Final earnings: ${result.earned_amount:,.2f}")
#             else:
#                 print(f"Earned so far: ${result.earned_amount:,.2f}")
#         else:
#             print("WRONG!")
#             print(f"Game Over. Final earnings: ${result.earned_amount:,.2f}")
#             break

#     print("\n=== Game Summary ===")
#     print(f"Reached Level: {game.current_level}")
#     print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [12]:
SPEECH_COMPETITION_ID = 1   # change to whichever competition you want

# `llm` and `strategy` are already built in the LlamaCpp setup section above.
# If you re-run only this section, uncomment the block below to recreate them.
llm = LlamaCppLLM(repo_id=DEFAULT_REPO_ID, filename=DEFAULT_FILENAME,
    model_path=GGUF_MODEL_PATH, max_new_tokens=64, temperature=0.0,
    n_ctx=4096, n_gpu_layers=-1, verbose=False)
strategy = GemmaStrategy(llm=llm)

print("\n=== Starting Speech Game ===")
speech_game = client.game.start(competition_id=SPEECH_COMPETITION_ID, mode="speech")
print(f"Session ID: {speech_game.session_id}  |  Mode: {speech_game.mode}")
print()
auto_play_speech(speech_game, strategy)

Loading existing local GGUF: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/llama-cpp/mistral-7b-instruct-v0.1.Q4_K_M.gguf
Initializing llama.cpp with model: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/llama-cpp/mistral-7b-instruct-v0.1.Q4_K_M.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized



=== Starting Speech Game ===
Session ID: 310658  |  Mode: speech



NameError: name 'auto_play_speech' is not defined

In speech mode, the 30 seconds timer is starting when we are requesting the last option!!

In [ ]:
# Show leaderboard position (speech mode)
lb = client.leaderboard.get(competition_id=SPEECH_COMPETITION_ID, limit=10, mode="speech") # <---- include mode!!
print(f"\n=== Speech Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")

## Results

In [ ]:
from pathlib import Path
from polimillionaire.runner import load_jsonl, summarize_attempts

print("Benchmark results:")
for row in benchmark_results:
    print(row)

live_logs = []
for item in run_results:
    for log_path in item.get("log_paths", []):
        live_logs.append(Path(log_path))
    if item.get("log_path"):
        live_logs.append(Path(item["log_path"]))
if live_logs:
    print("\nLive runs from this execution:")
    for latest in live_logs:
        print(latest.name)
        print(summarize_attempts(load_jsonl(latest)))
else:
    print("\nNo live game was run in this notebook execution.")
    old_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
    if old_logs:
        print("Most recent saved live log, not from this run:", old_logs[-1].name)
